In [65]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/frontal_cortex")

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/code/enrichment_fxns.R")

Here I perform enrichment analysis to find modules enriched for cell type markers. 

These modules will later be used to correlate with exon PSI to define cell type-specific exons.

In [66]:
mod_def <- "PosBC"

# Prep marker genes

## Pseudobulk DE genes

In [42]:
unique <- FALSE

In [43]:
# pairwise_res_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/tasic_2018/mouse_ALM/data/tasic_2018_ALM_STAR_donor_cell_type_pseudobulk_pairwise_DE_genes_dream.RDS")
# pairwise_ctype_genes <- prep_DE_genes(pairwise_res_list, lfc_threshold, pairwise=TRUE, unique=unique)

pairwise_res_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/ACA/data/DE_genes/yao_2021_ACA_STAR_donor_cell_type_pseudobulk_pairwise_DE_genes_dream.RDS")
pairwise_ctype_genes <- prep_DE_genes(pairwise_res_list, lfc_threshold, pairwise=TRUE, unique=unique)

marker_genes_list <- pairwise_ctype_genes

marker_genes_list <- lapply(marker_genes_list, toupper)

## MO marker genes

In [72]:
load("/mnt/lareaulab/reliscu/data/gene_sets/Oldham/MO_sets.RData")

In [ ]:
sets <- c(
    "BARRES_ASTROCYTES",
    "BARRES_OLIGODENDROCYTES",
    "HICKMAN_MICROGLIA_SENSOME_99",
    "BARRES_NEURONS",
    "BUTLER_ENDOTHELIAL_HPA",
    "ZEISEL_MURAL",
    "ZEISEL_EPENDYMAL",
    "ZEISEL_INTERNEURON",
    "ZHANG_MOUSE_OPC_2014",
    "Mukamel_InhibitoryNeurons",
    "Mukamel_ExcitatoryNeurons",
    "SUGINO_UP_GABAERGIC_NEURONS",
    "SUGINO_UP_GLUTAMATERGIC_NEURONS",
    "SUGINO_UP_LAYERS4-6_GABAERGIC_NEURONS_CINGULATE_CTX",
    "BAKKEN_2019_PVALB_GABAERGIC_DE_GABA_CLUSTERS",
    "BAKKEN_2019_VIP_GABAERGIC_DE_GABA_CLUSTERS",
    "BAKKEN_2019_SST_GABAERGIC_DE_GABA_CLUSTERS",
    "BAKKEN_2019_LAMP5_GABAERGIC_DE_GABA_CLUSTERS",
    "BAKKEN_2019_SNCG_GABAERGIC_DE_GABA_CLUSTERS",
    "BAKKEN_2019_SST_CHODL_GABAERGIC_DE_GABA_CLUSTERS",
    "VELMESHEV_2019_IN_SST",
    MO_legend$SetName[grepl("schirmer", MO_legend$SetName, ignore.case=T)],
    MO_legend$SetName[grepl("yang_pfc", MO_legend$SetName, ignore.case=T)]
)

In [75]:
mask <- MO_legend$SetName %in% sets

marker_genes_list <- MO_sets[mask]
names(marker_genes_list) <- MO_legend$SetName[mask] 

marker_genes_list <- lapply(marker_genes_list, toupper)

# Get enrichments

In [76]:
set_source <- "MO"

In [ ]:
network_dir <- "GTEx_frontal_cortex_TPM_OR_lmFit_residuals_mergeParam0.85_Modules"

In [82]:
enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

In [83]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)

In [80]:
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
YANG_PFC_2021_OLIGODENDROCYTE,1.853927e-277,2.079661e-272,yellow,Bicor-None_signum0.677_minSize15_merge_ME_0.85_18656
YANG_PFC_2021_ASTROCYTE,5.705126e-271,3.199891e-266,blue,Bicor-None_signum0.545_minSize15_merge_ME_0.85_18656
YANG_PFC_2021_MICROGLIA,1.252676e-191,4.684005e-188,green,Bicor-None_signum0.677_minSize10_merge_ME_0.85_18656
YANG_PFC_2021_ENDOTHELIAL,2.307534e-191,8.349997e-188,lightyellow,Bicor-None_signum0.545_minSize15_merge_ME_0.85_18656
YANG_PFC_2021_NRGN_NEURON,1.625261e-175,4.239891e-172,turquoise,Bicor-None_signum0.832_minSize10_merge_ME_0.85_18656
YANG_PFC_2021_L2_3_EXCITATORY_NEURON,1.064338e-142,1.456015e-139,turquoise,Bicor-None_signum0.832_minSize6_merge_ME_0.85_18656
YANG_PFC_2021_L4_EXCITATORY_NEURON,1.356634e-108,1.278838e-105,turquoise,Bicor-None_signum0.832_minSize4_merge_ME_0.85_18656
YANG_PFC_2021_L5_6_CC_EXCITATORY_NEURON,7.073673e-102,6.199190e-99,turquoise,Bicor-None_signum0.545_minSize15_merge_ME_0.85_18656
BUTLER_ENDOTHELIAL_HPA,7.633438e-90,5.335167e-87,salmon,Bicor-None_signum0.502_minSize12_merge_ME_0.85_18656


In [86]:
write.csv(top_mods_df, file=paste0("data/enrichments/GTEx_frontal_cortex_TPM_OR_lmFit_residuals_mergeParam0.85_top_", set_source, "_Qval_modules.csv"), row.names=FALSE, quote=FALSE)